# Thermal Expansion Prediction with SVM
## Gantry Setup with 4 Temperature Sensors and Camera Measurements

This notebook tests Support Vector Machine (SVM) models for predicting thermal expansion effects in a gantry system based on:
- Temperature data from 4 sensors
- Camera-based position measurements
- Thermal expansion compensation

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")


In [ ]:
# loding and visualizing the data

In [4]:
# Generate synthetic data for gantry setup with 4 temperature sensors
# In real implementation, this would be replaced with actual sensor data

np.random.seed(42)
n_samples = 1000

# Temperature sensors (4 sensors at different positions on gantry)
temp_sensor_1 = np.random.normal(25, 5, n_samples)  # Base temperature
temp_sensor_2 = temp_sensor_1 + np.random.normal(0, 1, n_samples)  # Correlated
temp_sensor_3 = temp_sensor_1 + np.random.normal(2, 1.5, n_samples)  # Slight offset
temp_sensor_4 = temp_sensor_1 + np.random.normal(-1, 1, n_samples)  # Another position

# Camera measurements (X, Y positions with thermal drift)
# Thermal expansion coefficient (typical for aluminum: ~23 µm/m/°C)
thermal_coeff = 23e-6  # m/m/°C
gantry_length = 1.0  # 1 meter gantry

# Base positions
x_base = np.random.uniform(-0.5, 0.5, n_samples)
y_base = np.random.uniform(-0.5, 0.5, n_samples)

# Temperature-induced expansion (simplified model)
avg_temp = (temp_sensor_1 + temp_sensor_2 + temp_sensor_3 + temp_sensor_4) / 4
temp_delta = avg_temp - 20  # Reference temperature 20°C

# Position errors due to thermal expansion
x_thermal_error = thermal_coeff * gantry_length * temp_delta * x_base
y_thermal_error = thermal_coeff * gantry_length * temp_delta * y_base

# Measured positions (with thermal drift and noise)
x_measured = x_base + x_thermal_error + np.random.normal(0, 0.001, n_samples)
y_measured = y_base + y_thermal_error + np.random.normal(0, 0.001, n_samples)

# Create DataFrame
data = pd.DataFrame({
    'temp_1': temp_sensor_1,
    'temp_2': temp_sensor_2,
    'temp_3': temp_sensor_3,
    'temp_4': temp_sensor_4,
    'x_base': x_base,
    'y_base': y_base,
    'x_measured': x_measured,
    'y_measured': y_measured,
    'x_error': x_measured - x_base,
    'y_error': y_measured - y_base
})

print(f"Generated dataset with {len(data)} samples")
print(f"Temperature range: {data[['temp_1', 'temp_2', 'temp_3', 'temp_4']].min().min():.1f}°C to {data[['temp_1', 'temp_2', 'temp_3', 'temp_4']].max().max():.1f}°C")
print(f"Position error range X: {data['x_error'].min()*1000:.3f} to {data['x_error'].max()*1000:.3f} mm")
print(f"Position error range Y: {data['y_error'].min()*1000:.3f} to {data['y_error'].max()*1000:.3f} mm")
data.head()

Generated dataset with 1000 samples
Temperature range: 6.9°C to 46.3°C
Position error range X: -2.903 to 2.927 mm
Position error range Y: -3.115 to 3.559 mm


,temp_1,temp_2,temp_3,temp_4,x_base,y_base,x_measured,y_measured,x_error,y_error
0,27.483571,28.882926,28.470803,24.575763,-0.014982,-0.331141,-0.015653,-0.331805,-0.000671,-0.000664
1,24.308678,25.233312,26.091900,22.448293,-0.414597,-0.039327,-0.414447,-0.040339,0.000150,-0.001012
2,28.238443,28.298073,29.049813,26.824837,0.472461,-0.192912,0.471201,-0.192694,-0.001260,0.000218
3,32.615149,31.968213,34.153207,33.502837,0.018010,-0.464841,0.018032,-0.466773,0.000021,-0.001933
4,23.829233,24.527456,22.988811,23.385786,0.114186,-0.176855,0.116096,-0.177158,0.001910,-0.000303


In [ ]:
# Visualize the data
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Temperature sensors correlation
axes[0, 0].scatter(data['temp_1'], data['temp_2'], alpha=0.6)
axes[0, 0].set_xlabel('Temperature Sensor 1 (°C)')
axes[0, 0].set_ylabel('Temperature Sensor 2 (°C)')
axes[0, 0].set_title('Temperature Sensor Correlation')

# Temperature vs X position error
avg_temp = data[['temp_1', 'temp_2', 'temp_3', 'temp_4']].mean(axis=1)
axes[0, 1].scatter(avg_temp, data['x_error']*1000, alpha=0.6, color='red')
axes[0, 1].set_xlabel('Average Temperature (°C)')
axes[0, 1].set_ylabel('X Position Error (mm)')
axes[0, 1].set_title('Temperature vs X Position Error')

# Temperature vs Y position error
axes[1, 0].scatter(avg_temp, data['y_error']*1000, alpha=0.6, color='green')
axes[1, 0].set_xlabel('Average Temperature (°C)')
axes[1, 0].set_ylabel('Y Position Error (mm)')
axes[1, 0].set_title('Temperature vs Y Position Error')

# Position error distribution
axes[1, 1].hist2d(data['x_error']*1000, data['y_error']*1000, bins=30, cmap='Blues')
axes[1, 1].set_xlabel('X Position Error (mm)')
axes[1, 1].set_ylabel('Y Position Error (mm)')
axes[1, 1].set_title('Position Error Distribution')

plt.tight_layout()
plt.show()

# Correlation matrix
correlation_matrix = data[['temp_1', 'temp_2', 'temp_3', 'temp_4', 'x_error', 'y_error']].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Matrix: Temperature Sensors vs Position Errors')
plt.show()

In [ ]:
# Prepare data for SVM training
# Features: 4 temperature sensors + base position
X = data[['temp_1', 'temp_2', 'temp_3', 'temp_4', 'x_base', 'y_base']].values
y_x = data['x_error'].values  # X position error
y_y = data['y_error'].values  # Y position error

# Split data into training and testing sets
X_train, X_test, y_x_train, y_x_test, y_y_train, y_y_test = train_test_split(
    X, y_x, y_y, test_size=0.2, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")

In [ ]:
# SVM Hyperparameter tuning for X position error
print("Training SVM for X position error prediction...")

# Parameter grid for grid search
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
    'kernel': ['rbf', 'poly', 'linear']
}

# Grid search for X position error
svm_x = SVR()
grid_search_x = GridSearchCV(svm_x, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid_search_x.fit(X_train_scaled, y_x_train)

print(f"Best parameters for X error: {grid_search_x.best_params_}")
print(f"Best CV score for X error: {-grid_search_x.best_score_:.6f}")

# Train final model for X
best_svm_x = grid_search_x.best_estimator_
y_x_pred = best_svm_x.predict(X_test_scaled)

# Evaluate X model
mse_x = mean_squared_error(y_x_test, y_x_pred)
r2_x = r2_score(y_x_test, y_x_pred)

print(f"\\nX Position Error Prediction:")
print(f"MSE: {mse_x:.8f} m²")
print(f"RMSE: {np.sqrt(mse_x)*1000:.3f} mm")
print(f"R² Score: {r2_x:.4f}")

In [ ]:
# SVM for Y position error
print("Training SVM for Y position error prediction...")

# Grid search for Y position error
grid_search_y = GridSearchCV(SVR(), param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid_search_y.fit(X_train_scaled, y_y_train)

print(f"Best parameters for Y error: {grid_search_y.best_params_}")
print(f"Best CV score for Y error: {-grid_search_y.best_score_:.6f}")

# Train final model for Y
best_svm_y = grid_search_y.best_estimator_
y_y_pred = best_svm_y.predict(X_test_scaled)

# Evaluate Y model
mse_y = mean_squared_error(y_y_test, y_y_pred)
r2_y = r2_score(y_y_test, y_y_pred)

print(f"\\nY Position Error Prediction:")
print(f"MSE: {mse_y:.8f} m²")
print(f"RMSE: {np.sqrt(mse_y)*1000:.3f} mm")
print(f"R² Score: {r2_y:.4f}")

In [ ]:
# Visualize prediction results
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# X position error: Actual vs Predicted
axes[0, 0].scatter(y_x_test*1000, y_x_pred*1000, alpha=0.6)
axes[0, 0].plot([y_x_test.min()*1000, y_x_test.max()*1000], 
                [y_x_test.min()*1000, y_x_test.max()*1000], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual X Error (mm)')
axes[0, 0].set_ylabel('Predicted X Error (mm)')
axes[0, 0].set_title(f'X Position Error Prediction (R² = {r2_x:.3f})')

# Y position error: Actual vs Predicted
axes[0, 1].scatter(y_y_test*1000, y_y_pred*1000, alpha=0.6)
axes[0, 1].plot([y_y_test.min()*1000, y_y_test.max()*1000], 
                [y_y_test.min()*1000, y_y_test.max()*1000], 'r--', lw=2)
axes[0, 1].set_xlabel('Actual Y Error (mm)')
axes[0, 1].set_ylabel('Predicted Y Error (mm)')
axes[0, 1].set_title(f'Y Position Error Prediction (R² = {r2_y:.3f})')

# Residuals for X
x_residuals = (y_x_test - y_x_pred) * 1000
axes[1, 0].scatter(y_x_pred*1000, x_residuals, alpha=0.6)
axes[1, 0].axhline(y=0, color='r', linestyle='--')
axes[1, 0].set_xlabel('Predicted X Error (mm)')
axes[1, 0].set_ylabel('Residuals (mm)')
axes[1, 0].set_title('X Error Residuals')

# Residuals for Y
y_residuals = (y_y_test - y_y_pred) * 1000
axes[1, 1].scatter(y_y_pred*1000, y_residuals, alpha=0.6)
axes[1, 1].axhline(y=0, color='r', linestyle='--')
axes[1, 1].set_xlabel('Predicted Y Error (mm)')
axes[1, 1].set_ylabel('Residuals (mm)')
axes[1, 1].set_title('Y Error Residuals')

plt.tight_layout()
plt.show()

# Summary of model performance
print("\\n" + "="*50)
print("THERMAL EXPANSION PREDICTION MODEL SUMMARY")
print("="*50)
print(f"Gantry Setup: 4 Temperature Sensors + Camera")
print(f"Dataset Size: {len(data)} samples")
print(f"Training Samples: {X_train.shape[0]}")
print(f"Testing Samples: {X_test.shape[0]}")
print("\\nX Position Error Model:")
print(f"  RMSE: {np.sqrt(mse_x)*1000:.3f} mm")
print(f"  R² Score: {r2_x:.4f}")
print(f"  Best Kernel: {grid_search_x.best_params_['kernel']}")
print("\\nY Position Error Model:")
print(f"  RMSE: {np.sqrt(mse_y)*1000:.3f} mm")
print(f"  R² Score: {r2_y:.4f}")
print(f"  Best Kernel: {grid_search_y.best_params_['kernel']}")
print("="*50)